## SCRIPT PYTHON TFG - CONSTRUCCIÓ DEL DATASET BASE

**Objectiu:**
1) Crear Datasetbase.xlsx amb una pestanya Dataset_base.
2) Incorporar les variants de mmc4 / Taula S5.
3) Consultar Ensembl GRCh37 per obtenir rsID, allele_string,
variant_class i al·lel de referència per posició hg19.
4) Convertir posicions hg19 a hg38.
5) Integrar al·lels neandertals i denisovans des de mmc3 / Taula S2.
6) Crear indicadors REF/HIII i HIII/Neandertal.
7) Crear la pestanya SNPs_main_analysis_clean amb els SNPs diferencials.
 
**IMPORTANT:**
- Aquest script NO inventa informació.
- Només extreu dades dels Excels d'entrada i de l'API d'Ensembl.
- Sempre usa la pestanya Dataset_base, no Dataset.
- No elimina pestanyes existents quan actualitza l'Excel.

In [ ]:

import os
import time
import requests
import pandas as pd

0. CONFIGURACIÓ DE FITXERS

output_file = "Datasetbase.xlsx"

dataset_sheet = "Dataset_base"
description_sheet = "Descripció_columnes"
manual_check_sheet = "Comprovacio_manual_rsID"
main_analysis_sheet = "SNPs_main_analysis_clean"

mmc4_file = "mmc4.xlsx"
sheet_mmc4 = "(a) Fst between continents"

mmc3_file = "mmc3.xlsx"
sheet_mmc3 = "Supplemental Table 2"

server_grch37 = "https://grch37.rest.ensembl.org"
server_current = "https://rest.ensembl.org"

headers_json = {"Content-Type": "application/json"}
headers_text = {"Content-Type": "text/plain"}



# FUNCIONS AUXILIARS

def write_sheet_excel(file_path, sheet_name, dataframe):
    """
    Escriu una pestanya a un Excel sense perdre les altres pestanyes.
    Si la pestanya ja existeix, la substitueix.
    """
    if os.path.exists(file_path):
        with pd.ExcelWriter(
            file_path,
            engine="openpyxl",
            mode="a",
            if_sheet_exists="replace"
        ) as writer:
            dataframe.to_excel(writer, sheet_name=sheet_name, index=False)
    else:
        with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
            dataframe.to_excel(writer, sheet_name=sheet_name, index=False)


def is_snv_allele_string(allele_string):
    """
    Retorna True si allele_string representa un SNV/SNP simple.
    Accepta: A/G, C/T, G/A/T.
    No accepta: A/AT, AT/A, -/A.
    """
    if allele_string is None or pd.isna(allele_string):
        return False

    alleles = str(allele_string).replace("|", "/").split("/")

    for allele in alleles:
        allele = allele.strip().upper()
        if allele not in ["A", "C", "G", "T"]:
            return False

    return True


def normalize_allele(value):
    """
    Normalitza un al·lel per poder comparar-lo.
    """
    if value is None or pd.isna(value):
        return None

    value = str(value).strip().upper()

    if value in ["", "NAN", "NONE"]:
        return None

    return value


def get_real_allele(allele, human_reference):
    """
    A mmc3, '.' indica identitat amb l'al·lel de referència humana.
    Aquesta funció substitueix '.' pel valor real de la referència.
    """
    allele = normalize_allele(allele)
    human_reference = normalize_allele(human_reference)

    if allele in [".", None]:
        return human_reference

    return allele


def request_with_retry(url, headers, max_retries=3, sleep_seconds=0.5):
    """
    Fa una consulta GET amb reintents bàsics.
    """
    for attempt in range(max_retries):
        try:
            r = requests.get(url, headers=headers, timeout=20)
            if r.ok:
                return r
        except requests.exceptions.RequestException:
            pass

        time.sleep(sleep_seconds)

    return None


def get_reference_by_position(chromosome, position):
    """
    Obté la base de referència humana segons la posició hg19/GRCh37.
    """
    chrom = str(chromosome).replace("chr", "").strip()
    pos = int(position)

    url = f"{server_grch37}/sequence/region/human/{chrom}:{pos}..{pos}:1"
    r = request_with_retry(url, headers=headers_text)

    if r is None:
        return None

    return r.text.strip().upper()


def get_hg38_position(chromosome, position):
    """
    Converteix una posició hg19/GRCh37 a hg38/GRCh38.
    """
    chrom = str(chromosome).replace("chr", "").strip()
    pos = int(position)

    url = f"{server_current}/map/human/GRCh37/{chrom}:{pos}..{pos}:1/GRCh38"
    r = request_with_retry(url, headers=headers_json)

    if r is None:
        return None

    data = r.json()
    mappings = data.get("mappings", [])

    if len(mappings) == 0:
        return None

    return mappings[0]["mapped"]["start"]


def get_snv_rsid(chromosome, position, hapIII_allele=None):
    """
    Busca variants en una posició hg19 i retorna un rsID de tipus SNP/SNV.
    Si hi ha més d'un SNP, prioritza el que contingui l'al·lel del haplotip III.
    """
    chrom = str(chromosome).replace("chr", "").strip()
    pos = int(position)

    url = f"{server_grch37}/overlap/region/human/{chrom}:{pos}-{pos}?feature=variation"
    r = request_with_retry(url, headers=headers_json)

    if r is None:
        return None, None, None

    variants = r.json()
    snv_candidates = []

    for variant in variants:
        rsid = variant.get("id")

        if rsid is None or not str(rsid).startswith("rs"):
            continue

        url_var = f"{server_grch37}/variation/human/{rsid}?content-type=application/json"
        r_var = request_with_retry(url_var, headers=headers_json)

        if r_var is None:
            continue

        data = r_var.json()
        variant_class = str(data.get("var_class", "")).lower()
        mappings = data.get("mappings", [])

        allele_string = None

        for m in mappings:
            if m.get("assembly_name") == "GRCh37":
                allele_string = m.get("allele_string")
                break

        if allele_string is None and len(mappings) > 0:
            allele_string = mappings[0].get("allele_string")

        # Només acceptem SNP/SNV simples
        if variant_class not in ["snp", "snv"]:
            continue

        if not is_snv_allele_string(allele_string):
            continue

        snv_candidates.append({
            "rsID": rsid,
            "allele_string": allele_string,
            "variant_class": "SNP"
        })

        time.sleep(0.1)

    if len(snv_candidates) == 0:
        return None, None, None

    # Prioritzar el rsID que conté l'al·lel del haplotip III
    hapIII_allele = normalize_allele(hapIII_allele)

    if hapIII_allele is not None:
        for candidate in snv_candidates:
            alleles = candidate["allele_string"].replace("|", "/").split("/")
            alleles = [a.strip().upper() for a in alleles]

            if hapIII_allele in alleles:
                return (
                    candidate["rsID"],
                    candidate["allele_string"],
                    candidate["variant_class"]
                )

    candidate = snv_candidates[0]
    return candidate["rsID"], candidate["allele_string"], candidate["variant_class"]

CREAR EXCEL BASE AMB ESTRUCTURA DE COLUMNES

In [ ]:
columnes = [
    "chr",
    "pos_hg19",
    "rsID",
    "Human_reference_by_position",
    "Allele_string_by_rsID",
    "Haplotype_V_allele",
    "Haplotype_III_allele",
    "Haplotype_IV_allele",
    "Haplotype_VII_allele",
    "Neandertal_allele",
    "Denisovan_allele",
    "III_differs_from_reference",
    "III_matches_Neandertal",
    "variant_class",
    "pos_hg38",
    "gene",
    "functional_region",
    "Use_in_main_analysis",
    "Notes"
]

column_description = pd.DataFrame({
    "Columna": columnes,
    "Descripció": [
        "Cromosoma en què es troba la variant",
        "Posició genòmica en hg19/GRCh37: coordenada original de l’article",
        "Identificador rsID de la variant",
        "Base de la referència humana obtinguda a partir de la posició genòmica",
        "Cadena d’al·lels associada al rsID segons dbSNP/Ensembl, útil per identificar SNPs o indels",
        "Al·lel present en el haplotip V",
        "Al·lel present en el haplotip III",
        "Al·lel present en el haplotip IV, informació contextual",
        "Al·lel present en el haplotip VII, informació contextual",
        "Al·lel present en el genoma de Neandertal segons l’article / validació",
        "Al·lel present en el genoma de Denisova segons l’article / validació",
        "Indica si el haplotip III difereix de la referència humana: Y/N/CHECK",
        "Indica si el haplotip III coincideix amb l’al·lel neandertal: Y/N/CHECK",
        "Classe de variant: SNP, indel, deletion, etc.",
        "Posició genòmica en hg38, si està disponible",
        "Gen associat a la variant",
        "Regió o conseqüència funcional segons anotació, per exemple intronic, upstream, missense",
        "Indica si la variant entra a l’anàlisi principal: Y/NO/CHECK",
        "Comentaris o decisions manuals"
    ]
})

# Es crea un Excel nou. Si ja existeix, se substitueix des de zero.
dataset = pd.DataFrame(columns=columnes)

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    dataset.to_excel(writer, sheet_name=dataset_sheet, index=False)
    column_description.to_excel(writer, sheet_name=description_sheet, index=False)

print(f"Fitxer base creat correctament: {output_file}")

INCORPORAR INFORMACIÓ DE MMC4 / TAULA S5

In [ ]:
dataset = pd.read_excel(output_file, sheet_name=dataset_sheet)
raw_mmc4 = pd.read_excel(mmc4_file, sheet_name=sheet_mmc4, header=None)

# Excel fila 3 = Python índex 2. S'agafen les 6 primeres columnes.
mmc4_data = raw_mmc4.iloc[2:, 0:6].copy()
mmc4_data.columns = [
    "chr",
    "pos_hg19",
    "Haplotype_V_allele",
    "Haplotype_III_allele",
    "Haplotype_IV_allele",
    "Haplotype_VII_allele"
]

mmc4_data = mmc4_data.dropna(subset=["chr", "pos_hg19"]).copy()

mmc4_data["chr"] = (
    mmc4_data["chr"]
    .astype(str)
    .str.replace("chr", "", regex=False)
    .str.strip()
)
mmc4_data["pos_hg19"] = mmc4_data["pos_hg19"].astype(int)

# Crear tantes files com variants hi ha a mmc4
dataset = pd.DataFrame(columns=columnes)
dataset = dataset.reindex(range(len(mmc4_data)))

print("Variants extretes de mmc4:", len(mmc4_data))

# Omplir columnes procedents de mmc4
dataset["chr"] = mmc4_data["chr"].values
dataset["pos_hg19"] = mmc4_data["pos_hg19"].values
dataset["Haplotype_V_allele"] = mmc4_data["Haplotype_V_allele"].values
dataset["Haplotype_III_allele"] = mmc4_data["Haplotype_III_allele"].values
dataset["Haplotype_IV_allele"] = mmc4_data["Haplotype_IV_allele"].values
dataset["Haplotype_VII_allele"] = mmc4_data["Haplotype_VII_allele"].values

write_sheet_excel(output_file, dataset_sheet, dataset)
print("Dataset actualitzat amb informació de mmc4.")
print(dataset.head())

CERCAR rsID, AL·LEL DE REFERÈNCIA, CLASSE I POSICIÓ hg38

In [ ]:
df = pd.read_excel(output_file, sheet_name=dataset_sheet)

new_rsids = []
allele_strings = []
variant_classes = []
reference_by_position = []
positions_hg38 = []

for index, row in df.iterrows():
    chrom = row["chr"]
    pos = row["pos_hg19"]
    hapIII = row["Haplotype_III_allele"]

    rsid, allele_string, variant_class = get_snv_rsid(chrom, pos, hapIII)
    ref_base = get_reference_by_position(chrom, pos)
    pos_hg38 = get_hg38_position(chrom, pos)

    new_rsids.append(rsid)
    allele_strings.append(allele_string)
    variant_classes.append(variant_class)
    reference_by_position.append(ref_base)
    positions_hg38.append(pos_hg38)

    print(
        f"chr{chrom}:{int(pos)} | "
        f"HIII={hapIII} | "
        f"rsID={rsid} | "
        f"alleles={allele_string} | "
        f"class={variant_class} | "
        f"REF_pos={ref_base} | "
        f"hg38={pos_hg38}"
    )

    time.sleep(0.2)


df["rsID"] = new_rsids
df["Allele_string_by_rsID"] = allele_strings
df["variant_class"] = variant_classes
df["Human_reference_by_position"] = reference_by_position
df["pos_hg38"] = positions_hg38

write_sheet_excel(output_file, dataset_sheet, df)
print("Dataset actualitzat amb rsID, allele_string, variant_class, referència per posició i posició hg38.")


MOSTRA ALEATÒRIA DE 10 rsID PER COMPROVACIÓ MANUAL

In [ ]:
df = pd.read_excel(output_file, sheet_name=dataset_sheet)

sample_df = df.sample(n=min(10, len(df)), random_state=42)
sample_df = sample_df[["chr", "pos_hg19", "rsID"]]

write_sheet_excel(output_file, manual_check_sheet, sample_df)

print("10 files seleccionades a l'atzar per comprovació manual:")
print(sample_df)

INTEGRAR AL·LELS NEANDERTALS I DENISOVANS DES DE MMC3 / TAULA S2

In [ ]:
dataset = pd.read_excel(output_file, sheet_name=dataset_sheet)
mmc3 = pd.read_excel(mmc3_file, sheet_name=sheet_mmc3)

# Netejar formats del dataset
dataset["chr"] = dataset["chr"].astype(str).str.replace("chr", "", regex=False).str.strip()
dataset["pos_hg19"] = dataset["pos_hg19"].astype(int)
dataset["rsID"] = dataset["rsID"].astype(str).str.strip()

# Netejar formats de mmc3
mmc3["Chromosome"] = mmc3["Chromosome"].astype(str).str.replace("chr", "", regex=False).str.strip()
mmc3["Position"] = mmc3["Position"].astype(int)
mmc3["SNP id"] = mmc3["SNP id"].astype(str).str.strip()

neandertal_alleles = []
denisovan_alleles = []
notes = []

for index, row in dataset.iterrows():
    chr_dataset = row["chr"]
    pos_dataset = row["pos_hg19"]
    rsid_dataset = row["rsID"]

    match = mmc3[
        (mmc3["Chromosome"] == chr_dataset) &
        (mmc3["Position"] == pos_dataset) &
        (mmc3["SNP id"] == rsid_dataset)
    ]

    if len(match) == 0:
        neandertal_alleles.append("R")
        denisovan_alleles.append("R")
        notes.append("Revisar manualment: no coincideix chr + pos_hg19 + rsID a mmc3")
        print(f"REVISAR: chr{chr_dataset}:{pos_dataset} {rsid_dataset}")
        continue

    matched_row = match.iloc[0]
    human_ref = matched_row["Human reference"]

    neandertal = get_real_allele(matched_row["I"], human_ref)
    denisova = get_real_allele(matched_row["II"], human_ref)

    neandertal_alleles.append(neandertal)
    denisovan_alleles.append(denisova)
    notes.append("Coincidència exacta a mmc3")

    print(
        f"Trobat: chr{chr_dataset}:{pos_dataset} {rsid_dataset} "
        f"| Neandertal={neandertal} | Denisova={denisova}"
    )


dataset["Neandertal_allele"] = neandertal_alleles
dataset["Denisovan_allele"] = denisovan_alleles

# Si Notes ja tenia alguna informació prèvia, aquí es prioritza la decisió de mmc3.
dataset["Notes"] = notes

CREAR INDICADORS REF/HIII I HIII/NEANDERTAL

In [ ]:
iii_differs = []
iii_matches_neandertal = []
use_in_main_analysis = []

for index, row in dataset.iterrows():
    ref = normalize_allele(row["Human_reference_by_position"])
    hapIII = normalize_allele(row["Haplotype_III_allele"])
    neandertal = normalize_allele(row["Neandertal_allele"])
    variant_class = normalize_allele(row["variant_class"])

    if ref is None or hapIII is None:
        iii_differs.append("CHECK")
        use_in_main_analysis.append("CHECK")
    elif hapIII != ref:
        iii_differs.append("Y")
        # Només entra a l'anàlisi principal si és SNP i HIII difereix de REF
        if variant_class == "SNP":
            use_in_main_analysis.append("Y")
        else:
            use_in_main_analysis.append("CHECK")
    else:
        iii_differs.append("N")
        use_in_main_analysis.append("NO")

    if neandertal in [None, "R"] or hapIII is None:
        iii_matches_neandertal.append("CHECK")
    elif hapIII == neandertal:
        iii_matches_neandertal.append("Y")
    else:
        iii_matches_neandertal.append("N")


dataset["III_differs_from_reference"] = iii_differs
dataset["III_matches_Neandertal"] = iii_matches_neandertal
dataset["Use_in_main_analysis"] = use_in_main_analysis

write_sheet_excel(output_file, dataset_sheet, dataset)

print("Dataset actualitzat amb al·lels de Neandertal i Denisova.")
print("Indicadors III_differs_from_reference, III_matches_Neandertal i Use_in_main_analysis creats.")
print("R = revisar manualment a mmc3 perquè no coincideix chr + pos_hg19 + rsID.")


CREAR PESTANYA AMB SNPs PRINCIPALS D'ANÀLISI

In [ ]:
snps_main = dataset[dataset["Use_in_main_analysis"] == "Y"].copy()

snps_main["Allelic_change_REF_to_HIII"] = (
    snps_main["Human_reference_by_position"].astype(str).str.upper()
    + ">"
    + snps_main["Haplotype_III_allele"].astype(str).str.upper()
)

write_sheet_excel(output_file, main_analysis_sheet, snps_main)

print("Pestanya SNPs_main_analysis_clean creada.")
print("Nombre de SNPs REF/HIII diferencials:", len(snps_main))

In [3]:
import pandas as pd
print(pd.__version__)

3.0.0
